## Main

### Import libraries

In [247]:
import pandas as pd
import numpy as np

# model selection
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

# model
from sklearn.linear_model import LogisticRegression

# evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# pipeline + imbalance handling
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV

### Parameters

In [248]:
DATA_PATH = "datasets/diabetic_data_imbalanced_0.965.csv"
TARGET_COLUMN = "diabetesmed"
RANDOM_STATE = 18

### Read data

In [249]:
df = pd.read_csv(DATA_PATH)
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,93499158,57292749,Caucasian,Male,[70-80),?,1,1,7,10,...,No,Steady,No,No,No,No,No,No,Yes,<30
1,149276196,27892332,AfricanAmerican,Female,[50-60),?,1,1,7,2,...,No,No,No,No,No,No,No,Ch,Yes,>30
2,154763322,65553282,Caucasian,Female,[90-100),?,1,3,7,3,...,No,No,No,No,No,No,No,No,Yes,NO
3,299038478,108728496,Caucasian,Male,[80-90),?,2,3,7,4,...,No,Steady,No,No,No,No,No,No,Yes,NO
4,267440622,57714192,Caucasian,Female,[70-80),?,2,1,1,14,...,No,Steady,No,No,No,No,No,Ch,Yes,<30


In [250]:
df.shape

(81205, 50)

In [251]:
df.groupby("diabetesMed")["patient_nbr"].count()

diabetesMed
No      2842
Yes    78363
Name: patient_nbr, dtype: int64

In [252]:
df.groupby("diabetesMed")["patient_nbr"].count() * 100 / df.shape[0]

diabetesMed
No      3.499784
Yes    96.500216
Name: patient_nbr, dtype: float64

### Data Preprocessing

In [253]:
clean_df = df.copy()

In [254]:
clean_df.columns = (
    clean_df.columns.str.lower().str.replace(" ", "_").str.replace("-", "_")
)
clean_df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'a1cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide_metformin', 'glipizide_metformin',
       'glimepiride_pioglitazone', 'metformin_rosiglitazone',
       'metformin_pioglitazone', 'change', 'diabetesmed', 'readmitted'],
      dtype='str')

In [255]:
clean_df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide_metformin,glipizide_metformin,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,change,diabetesmed,readmitted
0,93499158,57292749,Caucasian,Male,[70-80),?,1,1,7,10,...,No,Steady,No,No,No,No,No,No,Yes,<30
1,149276196,27892332,AfricanAmerican,Female,[50-60),?,1,1,7,2,...,No,No,No,No,No,No,No,Ch,Yes,>30
2,154763322,65553282,Caucasian,Female,[90-100),?,1,3,7,3,...,No,No,No,No,No,No,No,No,Yes,NO
3,299038478,108728496,Caucasian,Male,[80-90),?,2,3,7,4,...,No,Steady,No,No,No,No,No,No,Yes,NO
4,267440622,57714192,Caucasian,Female,[70-80),?,2,1,1,14,...,No,Steady,No,No,No,No,No,Ch,Yes,<30


### Data Exploration

In [256]:
for column in clean_df.columns:
    print()
    print("==" * 20)
    print("FEATURE NAME:", column)
    print()
    print("Uniqe values:")
    print(clean_df[column].unique())
    print()
    print("Value counts:")
    print(clean_df[column].value_counts())


FEATURE NAME: encounter_id

Uniqe values:
[ 93499158 149276196 154763322 ... 417962378   7977348  75109110]

Value counts:
encounter_id
93499158     1
149276196    1
154763322    1
299038478    1
267440622    1
            ..
37946328     1
211832892    1
417962378    1
7977348      1
75109110     1
Name: count, Length: 81205, dtype: int64

FEATURE NAME: patient_nbr

Uniqe values:
[57292749 27892332 65553282 ... 27646461  4519683 35387685]

Value counts:
patient_nbr
88785891    39
43140906    27
1660293     23
23643405    22
92709351    21
            ..
96095241     1
81811314     1
27646461     1
4519683      1
35387685     1
Name: count, Length: 58276, dtype: int64

FEATURE NAME: race

Uniqe values:
<ArrowStringArray>
['Caucasian', 'AfricanAmerican', 'Hispanic', '?', 'Asian', 'Other']
Length: 6, dtype: str

Value counts:
race
Caucasian          60597
AfricanAmerican    15344
?                   1913
Hispanic            1612
Other               1245
Asian                494
Name: co

### Remove columns with only one unique value

In [257]:
clean_df = clean_df.loc[:, clean_df.nunique() > 1]
clean_df.shape

(81205, 48)

### Remove columns with missing values ratio

In [258]:
MISSING_THRESHOLD = 0.5

clean_df = clean_df.replace("?", np.nan)
clean_df = clean_df.loc[:, clean_df.isna().mean() < MISSING_THRESHOLD]

clean_df.shape

(81205, 45)

### Remove ID columns

In [259]:
ID_COLUMN = ["encounter_id", "patient_nbr"]

clean_df = clean_df.drop(columns=ID_COLUMN, errors="ignore")
clean_df.shape

(81205, 43)

### Split Train Validation Test

In [260]:
clean_df.groupby("diabetesmed").count()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,troglitazone,tolazamide,insulin,glyburide_metformin,glipizide_metformin,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,change,readmitted
diabetesmed,,,,,,,,,,,,,,,,,,,,,
No,2800,2842,2842,2842,2842,2842,2842,1514,1398,2842,...,2842,2842,2842,2842,2842,2842,2842,2842,2842,2842
Yes,76492,78363,78363,78363,78363,78363,78363,48949,40396,78363,...,78363,78363,78363,78363,78363,78363,78363,78363,78363,78363


In [261]:
# Features
X = clean_df.drop(columns=["diabetesmed"])

# Target
y = clean_df["diabetesmed"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (81205, 42)
y shape: (81205,)


In [262]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,  # good for classification if classes are imbalanced
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (64964, 42)
y_train shape: (64964,)
X_test shape: (16241, 42)
y_test shape: (16241,)


In [263]:
y_train.value_counts(normalize=True)

diabetesmed
Yes    0.964996
No     0.035004
Name: proportion, dtype: float64

### Categorize data

In [264]:
for column in X_train.columns:
    print()
    print("==" * 20)
    print("FEATURE NAME:", column)
    print()
    print("Uniqe values:")
    print(X_train[column].unique())
    print()
    print("Value counts:")
    print(X_train[column].value_counts())


FEATURE NAME: race

Uniqe values:
<ArrowStringArray>
['Caucasian', 'AfricanAmerican', 'Hispanic', 'Asian', 'Other', nan]
Length: 6, dtype: str

Value counts:
race
Caucasian          48499
AfricanAmerican    12267
Hispanic            1281
Other                989
Asian                392
Name: count, dtype: int64

FEATURE NAME: gender

Uniqe values:
<ArrowStringArray>
['Male', 'Female', 'Unknown/Invalid']
Length: 3, dtype: str

Value counts:
gender
Female             34746
Male               30217
Unknown/Invalid        1
Name: count, dtype: int64

FEATURE NAME: age

Uniqe values:
<ArrowStringArray>
[ '[50-60)',  '[70-80)',  '[80-90)',  '[60-70)',  '[40-50)',  '[20-30)',
 '[90-100)',  '[30-40)',  '[10-20)',   '[0-10)']
Length: 10, dtype: str

Value counts:
age
[70-80)     16764
[60-70)     14552
[50-60)     11088
[80-90)     10727
[40-50)      6155
[30-40)      2374
[90-100)     1652
[20-30)      1060
[10-20)       482
[0-10)        110
Name: count, dtype: int64

FEATURE NAME: admissio

In [265]:
# ORDER_CATEGORIES = [
#     # age
#     [
#         "[0-10)",
#         "[10-20)",
#         "[20-30)",
#         "[30-40)",
#         "[40-50)",
#         "[50-60)",
#         "[60-70)",
#         "[70-80)",
#         "[80-90)",
#         "[90-100)",
#     ],
#     # medication columns
#     ["No", "Steady", "Up", "Down"],  # metformin
#     ["No", "Steady", "Up", "Down"],  # repaglinide
#     ["No", "Steady", "Up", "Down"],  # nateglinide
#     ["No", "Steady", "Up", "Down"],  # chlorpropamide
#     ["No", "Steady", "Up", "Down"],  # glimepiride
#     ["No", "Steady", "Up", "Down"],  # glipizide
#     ["No", "Steady", "Up", "Down"],  # glyburide
#     ["No", "Steady", "Up", "Down"],  # pioglitazone
#     ["No", "Steady", "Up", "Down"],  # rosiglitazone
#     ["No", "Steady", "Up", "Down"],  # acarbose
#     ["No", "Steady", "Up", "Down"],  # miglitol
#     # this one does not contain Down
#     ["No", "Steady", "Up"],  # tolazamide
#     ["No", "Steady", "Up", "Down"],  # insulin
#     ["No", "Steady", "Up", "Down"],  # glyburide_metformin
#     # target-style ordinal
#     ["NO", ">30", "<30"],  # readmitted
# ]

In [266]:
ORDER_CATEGORIES = [
    # age
    [
        "[0-10)",
        "[10-20)",
        "[20-30)",
        "[30-40)",
        "[40-50)",
        "[50-60)",
        "[60-70)",
        "[70-80)",
        "[80-90)",
        "[90-100)",
    ],
    # readmitted
    ["NO", ">30", "<30"],
    # change
    ["No", "Ch"],
]

In [267]:
# NUMERIC_COLUMNS = [
#     "time_in_hospital",
#     "num_lab_procedures",
#     "num_procedures",
#     "num_medications",
#     "number_outpatient",
#     "number_emergency",
#     "number_inpatient",
#     "number_diagnoses",
# ]

# NON_ORDER_CATEGORY_COLUMNS = [
#     "race",
#     "gender",
#     "admission_type_id",
#     "discharge_disposition_id",
#     "admission_source_id",
#     "payer_code",
#     "medical_specialty",
#     "diag_1",
#     "diag_2",
#     "diag_3",
#     "acetohexamide",
#     "tolbutamide",
#     "troglitazone",
#     "glipizide_metformin",
#     "glimepiride_pioglitazone",
#     "metformin_rosiglitazone",
#     "metformin_pioglitazone",
#     "change",
# ]

# ORDER_CATEGORY_COLUMNS = [
#     "age",
#     "metformin",
#     "repaglinide",
#     "nateglinide",
#     "chlorpropamide",
#     "glimepiride",
#     "glipizide",
#     "glyburide",
#     "pioglitazone",
#     "rosiglitazone",
#     "acarbose",
#     "miglitol",
#     "tolazamide",
#     "insulin",
#     "glyburide_metformin",
#     "readmitted",
# ]

# print(f"NUMERIC_COLUMNS: {len(NUMERIC_COLUMNS)}")
# print(f"NON_ORDER_CATEGORY_COLUMNS: {len(NON_ORDER_CATEGORY_COLUMNS)}")
# print(f"ORDER_CATEGORY_COLUMNS: {len(ORDER_CATEGORY_COLUMNS)}")
# print(f"X_train columns: {len(X_train.columns)}")
# print(
#     len(NUMERIC_COLUMNS) + len(NON_ORDER_CATEGORY_COLUMNS) + len(ORDER_CATEGORY_COLUMNS)
#     == len(X_train.columns)
# )

# all_defined_columns = (
#     NUMERIC_COLUMNS + NON_ORDER_CATEGORY_COLUMNS + ORDER_CATEGORY_COLUMNS
# )

# missing_columns = [col for col in X_train.columns if col not in all_defined_columns]

# print("Columns in X_train but not in the 3 sets:")
# print(missing_columns)
# print("Count:", len(missing_columns))

In [268]:
NUMERIC_COLUMNS = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

NON_ORDER_CATEGORY_COLUMNS = [
    "race",
    "gender",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
]

ORDER_CATEGORY_COLUMNS = [
    "age",
    "readmitted",
    "change",
]

print(f"NUMERIC_COLUMNS: {len(NUMERIC_COLUMNS)}")
print(f"NON_ORDER_CATEGORY_COLUMNS: {len(NON_ORDER_CATEGORY_COLUMNS)}")
print(f"ORDER_CATEGORY_COLUMNS: {len(ORDER_CATEGORY_COLUMNS)}")
print(f"X_train columns: {len(X_train.columns)}")
print(
    len(NUMERIC_COLUMNS) + len(NON_ORDER_CATEGORY_COLUMNS) + len(ORDER_CATEGORY_COLUMNS)
    == len(X_train.columns)
)

all_defined_columns = (
    NUMERIC_COLUMNS + NON_ORDER_CATEGORY_COLUMNS + ORDER_CATEGORY_COLUMNS
)

missing_columns = [col for col in X_train.columns if col not in all_defined_columns]

print("Columns in X_train but not in the 3 sets:")
print(missing_columns)
print("Count:", len(missing_columns))

NUMERIC_COLUMNS: 8
NON_ORDER_CATEGORY_COLUMNS: 10
ORDER_CATEGORY_COLUMNS: 3
X_train columns: 42
False
Columns in X_train but not in the 3 sets:
['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide_metformin', 'glipizide_metformin', 'glimepiride_pioglitazone', 'metformin_rosiglitazone', 'metformin_pioglitazone']
Count: 21


In [269]:
X_train = X_train[all_defined_columns]
X_train.shape

(64964, 21)

In [270]:
y_train.shape

(64964,)

### Train Model - Basic LogisticRegression

In [271]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# encode target
y_train = y_train.map({"Yes": 1, "No": 0})
y_test = y_test.map({"Yes": 1, "No": 0})

# preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_COLUMNS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), NON_ORDER_CATEGORY_COLUMNS),
        ("ord", OrdinalEncoder(categories=ORDER_CATEGORIES), ORDER_CATEGORY_COLUMNS),
    ]
)

# pipeline
pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced")),
    ]
)

# grid parameters
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 50],
    "model__solver": ["lbfgs", "liblinear"],
    "model__penalty": ["l2"]
}

# grid search
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

# train
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV ROC AUC:", grid.best_score_)

# best model
best_model = grid.best_estimator_

# predict
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# evaluation
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nROC AUC:", roc_auc_score(y_test, y_prob))

Fitting 5 folds for each of 10 candidates, totalling 50 fits


d:\GIT\diabetes_classification\mt_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best parameters: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV ROC AUC: 0.8410487502607455
Confusion Matrix:
 [[ 560    8]
 [5880 9793]]

Classification Report:
               precision    recall  f1-score   support

           0       0.09      0.99      0.16       568
           1       1.00      0.62      0.77     15673

    accuracy                           0.64     16241
   macro avg       0.54      0.81      0.46     16241
weighted avg       0.97      0.64      0.75     16241


ROC AUC: 0.8415754688919583
